# Inference time reasoning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shoaibphysics/reasoning-models/blob/main/notebooks/01_inference_time_reasoning.ipynb)

How do prompting and answer selection affect a language model's answers?

We study these choices using a fixed pretrained model. Its weights remain
unchanged throughout the experiments.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name,dtype=torch.float32).to(device)

model.eval()
print("Model loaded on:", device)

/Users/shoaibakhtar/Developer/reasoning-models/reasoning-models/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 536.77it/s]


Model loaded on: mps


### Text generation

By default, generation uses greedy decoding: each new token is the
highest-probability choice given the preceding tokens. Sampling can
be enabled to generate alternative responses.

In [2]:
@torch.inference_mode()
def ask(prompt, max_new_tokens=256, sample=False):
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt").to(device)

    options = {}
    if sample:
        options = {
            "temperature": 0.7,
            "top_p": 0.9,
            "top_k": 0 # Disable top-k filtering; top-p determines which tokens can be sampled.
            }

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=sample,
        pad_token_id=tokenizer.eos_token_id,
        **options)

    prompt_length = inputs["input_ids"].shape[1]
    new_tokens = output[0, prompt_length:]

    return tokenizer.decode(new_tokens, skip_special_tokens=True)

### Direct-answer baseline

We ask three arithmetic questions using the same instruction: return only
the final numerical answer.

The expected answers are used for evaluation and are not included in the
prompts. This small set illustrates model behavior rather than measuring
general reasoning ability.

In [3]:
problems = [
    ("Mira has 7 marbles. She buys 3 bags with 4 marbles each, then gives away 5 marbles. How many remain?",
        14),
    ("A notebook costs $6. A pen costs $3. How much do 2 notebooks and 3 pens cost?",
        21),
    ("A bus has 18 passengers. At a stop, 7 leave and 4 board. How many passengers are now on the bus?",
        15)]

baseline_answers = []

for question, expected in problems:
    prompt = question + "\nGive only the final numerical answer."
    response = ask(prompt)
    baseline_answers.append(response)

    print("Question:", question)
    print("Model:", response)
    print("Expected:", expected)
    print()

Question: Mira has 7 marbles. She buys 3 bags with 4 marbles each, then gives away 5 marbles. How many remain?
Model: 12
Expected: 14

Question: A notebook costs $6. A pen costs $3. How much do 2 notebooks and 3 pens cost?
Model: 15
Expected: 21

Question: A bus has 18 passengers. At a stop, 7 leave and 4 board. How many passengers are now on the bus?
Model: 20
Expected: 15



### Step-by-step prompting

A step-by-step prompt asks the model to generate intermediate calculations
before giving its final answer.

We compare this with direct-answer prompting on the same questions.
Both the calculations and the final answers must be checked: a longer
response is not necessarily more accurate.

In [4]:
step_by_step_answers = []

for i, (question, expected) in enumerate(problems):
    prompt = (question
              + "\nSolve this step by step. Keep the explanation brief."
              + "\nEnd with 'Final answer: <number>'.")

    response = ask(prompt)
    step_by_step_answers.append(response)

    print("Question:", question)
    print("Direct answer:", baseline_answers[i])
    print("Step-by-step response:")
    print(response)
    print("Expected:", expected)
    print("-" * 60)

Question: Mira has 7 marbles. She buys 3 bags with 4 marbles each, then gives away 5 marbles. How many remain?
Direct answer: 12
Step-by-step response:
Let's solve this problem step by step:

1. **Initial number of marbles**: Mira starts with 7 marbles.

2. **Marbles bought in bags**: She buys 3 bags, and each bag contains 4 marbles. So, she gets:
   \[
   3 \text{ bags} \times 4 \text{ marbles per bag} = 12 \text{ marbles}
   \]

3. **Total marbles after buying**: Adding the marbles from the bags to her initial amount:
   \[
   7 \text{ (initial)} + 12 \text{ (bought)} = 19 \text{ marbles}
   \]

4. **Marbles given away**: Mira gives away 5 marbles. Subtracting these from the total:
   \[
   19 \text{ (total)} - 5 \text{ (given away)} = 14 \text{ marbles remaining}
   \]

So, the final answer is:
\[
\boxed{14}
\]
Expected: 14
------------------------------------------------------------
Question: A notebook costs $6. A pen costs $3. How much do 2 notebooks and 3 pens cost?
Direct answe

### Few-shot chain-of-thought prompting

Few-shot prompting includes example questions and answers in the prompt.
Here, each example demonstrates a short calculation followed by a
consistently formatted final answer.

The demonstration questions are separate from the evaluation questions.

In [5]:
examples = """Question: A plate has 8 cookies. We add 5 cookies, then eat 3. How many remain?
Solution: 8 + 5 - 3 = 10.
Final answer: 10

Question: Three tickets cost $4 each, and two drinks cost $3 each. What is the total cost?
Solution: The tickets cost 3 * 4 = 12 dollars. The drinks cost 2 * 3 = 6 dollars. The total is 12 + 6 = 18 dollars.
Final answer: 18"""

In [6]:
few_shot_answers = []

for question, expected in problems:
    prompt = (
        examples
        + "\n\nQuestion: " + question
        + "\nSolve this step by step. Keep the explanation brief."
        + "\nEnd with 'Final answer: <number>'.")

    response = ask(prompt)
    few_shot_answers.append(response)

    print("Question:", question)
    print("Few-shot response:")
    print(response)
    print("Expected:", expected)
    print("-" * 60)

Question: Mira has 7 marbles. She buys 3 bags with 4 marbles each, then gives away 5 marbles. How many remain?
Few-shot response:
Let's solve it step by step:

1. Initially, Mira has 7 marbles.
2. She buys 3 bags with 4 marbles each:
   \[
   3 \times 4 = 12 \text{ marbles}
   \]
3. After buying the marbles, she has:
   \[
   7 + 12 = 19 \text{ marbles}
   \]

4. Then, she gives away 5 marbles:
   \[
   19 - 5 = 14 \text{ marbles}
   \]

So, the final number of marbles remaining is **14**.

Final answer: 14
Expected: 14
------------------------------------------------------------
Question: A notebook costs $6. A pen costs $3. How much do 2 notebooks and 3 pens cost?
Few-shot response:
Let's solve it step by step:

1. First, we need to find out how much one notebook costs:
   - One notebook costs $6.

2. Next, we calculate the cost of 2 notebooks:
   - Cost of 2 notebooks = 2 × $6 = $12.

3. Now, let's find out how much one pen costs:
   - One pen costs $3.

4. Finally, we calculate the

### Few-shot direct prompting

The demonstrations contain final answers without intermediate calculations.
We use the same demonstration questions as in few-shot chain-of-thought
prompting, keeping the model and evaluation questions fixed.

In [7]:
direct_examples = """Question: A plate has 8 cookies. We add 5 cookies, then eat 3. How many remain?
Final answer: 10

Question: Three tickets cost $4 each, and two drinks cost $3 each. What is the total cost?
Final answer: 18"""

few_shot_direct_answers = []

for question, expected in problems:
    prompt = (
        direct_examples
        + "\n\nQuestion: " + question
        + "\nGive only the final numerical answer."
        + "\nUse the format 'Final answer: <number>'."
    )

    response = ask(prompt)
    few_shot_direct_answers.append(response)

    print("Question:", question)
    print("Few-shot direct response:", response)
    print("Expected:", expected)
    print()

Question: Mira has 7 marbles. She buys 3 bags with 4 marbles each, then gives away 5 marbles. How many remain?
Few-shot direct response: Final answer: 2
Expected: 14

Question: A notebook costs $6. A pen costs $3. How much do 2 notebooks and 3 pens cost?
Few-shot direct response: Final answer: 9
Expected: 21

Question: A bus has 18 passengers. At a stop, 7 leave and 4 board. How many passengers are now on the bus?
Few-shot direct response: Final answer: 19
Expected: 15



### Sampling multiple solutions

Sampling selects each next token probabilistically, allowing repeated
generation from the same prompt to produce different solutions.

Different responses may reach the same final answer. Agreement does not
guarantee correctness.

In [8]:
torch.manual_seed(42)

sample_question, sample_expected = problems[0]

prompt = (
    examples
    + "\n\nQuestion: " + sample_question
    + "\nSolve this step by step. Keep the explanation brief."
    + "\nEnd with 'Final answer: <number>'.")

samples = []

for i in range(5):
    response = ask(prompt, sample=True)
    samples.append(response)

    print(f"Sample {i + 1}:")
    print(response)
    print("-" * 60)

Sample 1:
Let's solve this step by step:

1. Initially, Mira has 7 marbles.

2. She buys 3 bags with 4 marbles each:
   \( 3 \times 4 = 12 \) more marbles.
   Now she has \( 7 + 12 = 19 \) marbles.

3. Then she gives away 5 marbles:
   \( 19 - 5 = 14 \).

So, the final number of marbles remaining is 14.

The final answer is: **14**.
------------------------------------------------------------
Sample 2:
To solve this problem, let's go through it step by step:

1. **Initial number of marbles:** Mira starts with 7 marbles.
2. **Marbles bought in one bag:** Each bag contains 4 marbles, and she buys 3 bags.
   \[
   3 \text{ bags} \times 4 \text{ marbles per bag} = 12 \text{ marbles}
   \]
3. **Total number of marbles after buying:** Adding the marbles from the bags to her initial amount.
   \[
   7 \text{ marbles} + 12 \text{ marbles} = 19 \text{ marbles}
   \]

4. **Marbles given away:** Mira gives away 5 marbles.
   \[
   19 \text{ marbles} - 5 \text{ marbles} = 14 \text{ marbles}
   \]


#### Majority voting

We extract the final integer answer from each sampled response and count
the votes.

An answer is selected only if it receives more than half of all sample
votes. Unreadable answers do not vote. The expected answer is used only
to evaluate the selection.

In [9]:
# extract the answers
import re
from collections import Counter

def extract_answer(text):
    text = text.replace("**", "")
    match = re.search(
        r"final answer(?:\s+is)?\s*:\s*\$?\s*(-?\d+)\s*[.!]?\s*$",
        text,
        flags=re.IGNORECASE,
    )
    return int(match.group(1)) if match else None

sampled_answers = [extract_answer(text) for text in samples]
print(sampled_answers)

[14, 14, 14, 14, 7]


In [10]:
counts = Counter(answer for answer in sampled_answers if answer is not None)

winner, votes = counts.most_common(1)[0] if counts else (None, 0)
selected_answer = winner if votes > len(samples) / 2 else None

print("Votes:", dict(counts))
print("Selected answer:", selected_answer)
print("Expected answer:", sample_expected)

Votes: {14: 4, 7: 1}
Selected answer: 14
Expected answer: 14


#### Best-of-N selection

A reward model assigns a numerical score to each question–response pair.
Best-of-N selects the candidate with the highest score.

We score the same candidates used for majority voting. The scores represent
learned answer preferences, not probabilities of mathematical correctness.
The expected answer is used only to evaluate the selected response.

In [11]:
from transformers import AutoModelForSequenceClassification

reward_model_name = "OpenAssistant/reward-model-deberta-v3-large"
reward_tokenizer = AutoTokenizer.from_pretrained(reward_model_name,use_fast=True)
reward_model = AutoModelForSequenceClassification.from_pretrained(reward_model_name,dtype=torch.float32).to("cpu").eval()
print("Reward model loaded.")

/Users/shoaibakhtar/Developer/reasoning-models/reasoning-models/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|██████████| 394/394 [00:00<00:00, 77193.38it/s]

Reward model loaded.


In [12]:
@torch.inference_mode()
def reward_score(question, response):
    inputs = reward_tokenizer(question,response,return_tensors="pt")
    if inputs["input_ids"].shape[1] > reward_model.config.max_position_embeddings:
        raise ValueError("Question and response exceed the scoring token limit.")
    return reward_model(**inputs).logits.item()

scores = [reward_score(sample_question, response) for response in samples]

for i, score in enumerate(scores):
    answer = extract_answer(samples[i])
    print(f"Sample {i + 1}: score={score:.3f}, answer={answer}")

best_index = max(range(len(scores)), key=lambda i: scores[i])
best_of_n_answer = extract_answer(samples[best_index])

print()
print("Selected sample:", best_index + 1)
print("Best-of-N answer:", best_of_n_answer)
print("Voting answer:", selected_answer)
print("Expected answer:", sample_expected)

Sample 1: score=3.168, answer=14
Sample 2: score=5.288, answer=14
Sample 3: score=2.276, answer=14
Sample 4: score=2.942, answer=14
Sample 5: score=3.572, answer=7

Selected sample: 2
Best-of-N answer: 14
Voting answer: 14
Expected answer: 14


Majority voting selected 14 with four of five votes. Best-of-N also
selected 14, choosing Sample 2.

Sample 5 omitted Mira's initial seven marbles from its calculation,
yet received the second-highest reward score. The ranking did not
consistently reflect arithmetic correctness.

### Reward-guided beam search

We generate several partial solutions using beam decoding.
For each solution, we average the reward scores of its prefixes
and keep the two highest-scoring candidates for the next round.

We start with four candidates and extend unfinished responses in
chunks of 64 tokens. Splitting the generated text at ". " provides
a rough approximation of reasoning steps.

In [13]:
search_prompt = (
    examples
    + "\n\nQuestion: " + sample_question
    + "\nSolve this step by step. Keep the explanation brief."
    + "\nEnd with 'Final answer: <number>'.")

search_inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": search_prompt}],
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt").to(device)

search_prompt_length = search_inputs["input_ids"].shape[1]

with torch.inference_mode():
    candidate_ids = model.generate(
        **search_inputs,
        max_new_tokens=64,
        num_beams=4,
        num_return_sequences=4,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id)

In [14]:
def prefix_score(question, response):
    pieces = response.split(". ")
    prefix_scores = [
        reward_score(question, ". ".join(pieces[:i]))
        for i in range(1, len(pieces) + 1)]
    return sum(prefix_scores) / len(prefix_scores)

candidates = []

for i, ids in enumerate(candidate_ids, start=1):
    response = tokenizer.decode(
        ids[search_prompt_length:],
        skip_special_tokens=True)
    score = prefix_score(sample_question, response)

    candidates.append({
        "number": i,
        "ids": ids.unsqueeze(0),
        "text": response,
        "score": score})

    print(f"Candidate {i} | score={score:.3f}")
    print(response)
    print("-" * 60)

beams = sorted(
    candidates,
    key=lambda candidate: candidate["score"],
    reverse=True)[:2]

print("Retained candidates:", [beam["number"] for beam in beams])

Candidate 1 | score=-1.318
Let's break it down step by step:

1. Mira starts with 7 marbles.
2. She buys 3 bags with 4 marbles each, so she buys a total of 3 * 4 = 12 marbles.
3. After buying the marbles, she has 7
------------------------------------------------------------
Candidate 2 | score=-0.800
Let's solve this step by step:

1. Mira starts with 7 marbles.
2. She buys 3 bags with 4 marbles each. So, she buys a total of 3 * 4 = 12 marbles.
3. After buying the marbles, she has 7
------------------------------------------------------------
Candidate 3 | score=-0.670
Let's solve this step by step:

1. Mira starts with 7 marbles.
2. She buys 3 bags with 4 marbles each. So, she buys 3 * 4 = 12 marbles.
3. After buying the marbles, she has 7 + 1
------------------------------------------------------------
Candidate 4 | score=-1.536
Let's break it down step by step:

1. Mira starts with 7 marbles.
2. She buys 3 bags with 4 marbles each, so she buys a total of 3 * 4 = 12 marbles.
3. Afte

In [15]:
eos_ids = model.generation_config.eos_token_id
if isinstance(eos_ids, int):
    eos_ids = [eos_ids]

def is_finished(candidate):
    generated = candidate["ids"][0, search_prompt_length:]
    return any(token in eos_ids for token in generated.tolist())

# Start from the original four candidates whenever this cell is rerun.
beams = sorted(candidates, key=lambda c: c["score"], reverse=True)[:2]
completed = [c for c in candidates if is_finished(c)]

for round_number in range(1, 4):
    if all(is_finished(beam) for beam in beams):
        break

    expanded = []

    for beam in beams:
        if is_finished(beam):
            expanded.append(beam)
            continue

        with torch.inference_mode():
            children = model.generate(
                input_ids=beam["ids"],
                attention_mask=torch.ones_like(beam["ids"]),
                max_new_tokens=64,
                num_beams=2,
                num_return_sequences=2,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id)

        for ids in children:
            response = tokenizer.decode(
                ids[search_prompt_length:],
                skip_special_tokens=True)

            child = {
                "ids": ids.unsqueeze(0),
                "text": response,
                "score": prefix_score(sample_question, response)}

            expanded.append(child)

            if is_finished(child):
                completed.append(child)

    beams = sorted(expanded,key=lambda c: c["score"],reverse=True)[:2]

    print(f"Expansion round {round_number}")
    for i, beam in enumerate(beams, start=1):
        print(f"  Beam {i}: score={beam['score']:.3f}, finished={is_finished(beam)}")

Expansion round 1
  Beam 1: score=0.377, finished=True
  Beam 2: score=0.337, finished=True


In [16]:
if completed:
    best_search_result = max(completed, key=lambda c: c["score"])

    print(best_search_result["text"])
    print()
    print("Search score:", round(best_search_result["score"], 3))
    print("Extracted answer:", extract_answer(best_search_result["text"]))
    print("Expected answer:", sample_expected)
else:
    print("No response reached an end token within the generation budget.")
    print("Highest-scoring retained partial response:")
    print(beams[0]["text"])

Let's solve this step by step:

1. Mira starts with 7 marbles.
2. She buys 3 bags with 4 marbles each. So, she buys a total of 3 * 4 = 12 marbles.
3. After buying the marbles, she has 7 + 12 = 19 marbles.
4. She then gives away 5 marbles. So, she has 19 - 5 = 14 marbles remaining.

Final answer: 14

Search score: 0.377
Extracted answer: 14
Expected answer: 14


### Results and observations

| Method | Questions evaluated | Correct answers |
|---|---:|---:|
| Zero-shot direct | 3 | 0 |
| Zero-shot chain-of-thought | 3 | 3 |
| Few-shot direct | 3 | 0 |
| Few-shot chain-of-thought | 3 | 3 |
| Majority voting | 1 | 1 |
| Best-of-N | 1 | 1 |
| Reward-guided beam search | 1 | 1 |

The prompting methods were evaluated on three arithmetic questions.
Voting, best-of-N, and beam search were demonstrated on the marbles
question only.

Both chain-of-thought prompting conditions answered all three questions
correctly. Providing answer-only demonstrations did not improve accuracy
on this set.

All three selection methods returned 14 for the marbles question.
However, the reward model ranked one incorrect sampled solution above
three correct ones, showing that its preferences do not consistently
track mathematical correctness.

These examples show how the methods work, but there are too few questions
to tell which method is most reliable. They also require different amounts
of computation: some generate a single answer, while others generate and
score several candidates.

Further experiments could compare different model sizes using prompts
with no examples, one example, or several examples, both with and without
step-by-step instructions. Testing on more questions would help show how
model size and prompting style affect accuracy.